# Машинное обучение: полный pipeline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np

# Загрузка данных
df = pd.read_csv("https://raw.githubusercontent.com/aiedu-courses/stepik_eda_and_dev_tools/main/datasets/online_shoppers_intention.csv")
print(f"Размер датасета: {df.shape}")
print(f"\nЦелевая переменная (Revenue):\n{df['Revenue'].value_counts(normalize=True)}")

Размер датасета: (12330, 18)

Целевая переменная (Revenue):
Revenue
False    0.845255
True     0.154745
Name: proportion, dtype: float64


**Числовые**:
Administrative,
Administrative_Duration,
Informational,
Informational_Duration,
ProductRelated,
ProductRelated_Duration,
BounceRates,
ExitRates,
PageValues,
SpecialDay.

**Категориальные**:
Month,
VisitorType,
Weekend,
Browser,
Region,
TrafficType,
OperatingSystems.

В датасете наблюдается выраженный дисбаланс классов (84.5% False против 15.5% True), из-за чего метрика accuracy_score является неинформативной, так как модель может получить высокий результат, просто всегда предсказывая мажоритарный класс.

In [ ]:
# Обработка пропусков медианой для числовых переменных
df.fillna({
    'Informational_Duration': df['Informational_Duration'].median(),
    'ProductRelated_Duration': df['ProductRelated_Duration'].median(),
    'ExitRates':df['ExitRates'].median()
}, inplace=True)

# Удаление дубликатов
df.drop_duplicates(inplace=True)
df.reset_index(inplace=True, drop=True)

df['Revenue'] = df['Revenue'].astype(int)

# Исправляю ошибку с августом из прошлого дз)
df['Month'] = df['Month'].str.lower().str.capitalize()

# Обработка пропусков модой для категориальных переменных
df['Browser'] = df['Browser'].fillna(df['Browser'].mode()[0])
df['Region'] = df['Region'].fillna(df['Region'].mode()[0])
df['TrafficType'] = df['TrafficType'].fillna(df['TrafficType'].mode()[0])


1. Построение моделей на числовых признаках (минимум - NB и kNN, максимум - любые) с параметрами по умолчанию, вычисление метрик

In [ ]:
# Выбор числовых признаков
numeric_features = [
    'Administrative', 'Administrative_Duration', 'Informational',
    'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay'
]


X = df[numeric_features]
y = df['Revenue']

# Разделение на тренировочную и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Naive Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

# kNN
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

# Функция для вывода метрик
def print_metrics(y_true, y_pred, model_name):
    print(f"\n{model_name}:")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall: {recall_score(y_true, y_pred):.4f}")
    print(f"F1-score: {f1_score(y_true, y_pred):.4f}")
    print(f"Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}")

print_metrics(y_test, y_pred_nb, "Naive Bayes")
print_metrics(y_test, y_pred_knn, "kNN")


Naive Bayes:
Accuracy: 0.8458
Precision: 0.4915
Recall: 0.5011
F1-score: 0.4963
Confusion Matrix:
[[2352  240]
 [ 231  232]]

kNN:
Accuracy: 0.8638
Precision: 0.6009
Recall: 0.3024
F1-score: 0.4023
Confusion Matrix:
[[2499   93]
 [ 323  140]]


Обе модели слабоваты из-за дисбаланса классов. Naive Bayes показывает более сбалансированный результат (F1=0.496), находя примерно половину реальных конверсий, тогда как kNN чаще ошибается: он точнее в прогнозах Precision=0.6, но пропускает более 70% реальных покупок Recall=0.3.

2. Подбор гиперпараметров с GridSearchCV

In [ ]:
# GridSearch для Naive Bayes
nb_params = {'var_smoothing': np.logspace(-9, -1, 9)}

gs_nb = GridSearchCV(GaussianNB(), nb_params, cv=5, scoring='f1', n_jobs=-1)
gs_nb.fit(X_train, y_train)

print(f"Best NB params: {gs_nb.best_params_}")
print(f"Best NB CV score: {gs_nb.best_score_:.4f}")

# GridSearch для kNN
knn_params = {
    'n_neighbors': np.arange(2, 30, 2),
    'weights': ['uniform', 'distance'],
     'p' : [1, 2]
}
gs_knn = GridSearchCV(KNeighborsClassifier(), knn_params, cv=5, scoring='f1', n_jobs=-1)
gs_knn.fit(X_train, y_train)

print(f"Best kNN params: {gs_knn.best_params_}")
print(f"Best kNN CV score: {gs_knn.best_score_:.4f}")

# Оценка лучших моделей на тесте
y_pred_nb_best = gs_nb.best_estimator_.predict(X_test)
y_pred_knn_best = gs_knn.best_estimator_.predict(X_test)

print_metrics(y_test, y_pred_nb_best, "Naive Bayes (tuned)")
print_metrics(y_test, y_pred_knn_best, "kNN (tuned)")

Best NB params: {'var_smoothing': np.float64(1e-09)}
Best NB CV score: 0.5182
Best kNN params: {'n_neighbors': np.int64(4), 'p': 2, 'weights': 'distance'}
Best kNN CV score: 0.4309

Naive Bayes (tuned):
Accuracy: 0.8458
Precision: 0.4915
Recall: 0.5011
F1-score: 0.4963
Confusion Matrix:
[[2352  240]
 [ 231  232]]

kNN (tuned):
Accuracy: 0.8556
Precision: 0.5379
Recall: 0.3369
F1-score: 0.4143
Confusion Matrix:
[[2458  134]
 [ 307  156]]


GridSearch почти не улучшил модели: Naive Bayes остался на том же уровне (F1=0.496), а kNN слегка подтянул метрики (Recall 0.30→0.34, F1 0.4→0.41), но всё ещё пропускает две трети реальных покупок. Naive Bayes предпочтительнее — он лучше находит целевые события, тогда как kNN по-прежнему жертвует полнотой ради точности. Нужен не только подбор параметров, но и масштабирование данных, балансировка классов или смена алгоритма.

3. Добавление категориальных признаков в лучшую модель, обучение модели и заново подбор ее гиперпараметров + вычисление метрик

In [ ]:
# Списки признаков
categorical_features = ['Month', 'VisitorType', 'Weekend',
                   'OperatingSystems', 'Browser', 'Region', 'TrafficType']
numeric_features = ['Administrative', 'Administrative_Duration', 'Informational',
                   'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
                   'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay']

# Данные и целевая переменная
X = df[categorical_features + numeric_features]
y = df['Revenue']

# Разделение
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(X, y, test_size=0.25, random_state=42)

# Преобразователь и пайплайн
preprocessor = ColumnTransformer([
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ('scaler', MinMaxScaler(), numeric_features)
])

# Пайплайн с лучшей моделью
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GaussianNB())
])

# Параметры для GridSearch
params = {
    'classifier__var_smoothing': np.logspace(-12, -6, 7)
}

gs_pipeline = GridSearchCV(pipeline, params, cv=5, scoring='f1', n_jobs=-1)
gs_pipeline.fit(X_train_full, y_train_full)

# Оценка на тесте
y_pred_best = gs_pipeline.best_estimator_.predict(X_test_full)
print_metrics(y_test_full, y_pred_best, "GaussianNB with categorical features (tuned)")


GaussianNB with categorical features (tuned):
Accuracy: 0.3218
Precision: 0.1786
Recall: 0.9654
F1-score: 0.3014
Confusion Matrix:
[[ 536 2056]
 [  16  447]]


После добавления категориальных признаков и OneHotEncoding качество GaussianNB ухудшилось. Несмотря на резкий рост recall, модель стала генерировать большое количество ложноположительных прогнозов, из-за чего precision, accuracy и F1-score существенно снизились.


In [ ]:
# Установка
!pip install -q explainerdashboard

import warnings
warnings.filterwarnings("ignore")

from explainerdashboard import ClassifierExplainer, ExplainerDashboard
from google.colab import output

# explainer
explainer = ClassifierExplainer(
    gs_pipeline.best_estimator_,
    X_test_full,
    y_test_full,
    labels=["No Revenue", "Revenue"],
)

db = ExplainerDashboard(explainer, mode='external')

# проксируем порт
output.serve_kernel_port_as_iframe(8050, height=900)

# запуск
db.run(port=8050)



  0%|          | 0/3055 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning:

X has feature names, but GaussianNB was fitted without feature names



<IPython.core.display.Javascript object>

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app 'explainerdashboard.dashboards'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [16/May/2026 15:19:26] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [16/May/2026 15:19:26] "GET /assets/bootstrap.min.css?m=1778938176.38604 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [16/May/2026 15:19:26] "GET /_dash-component-suites/dash/deps/prop-types@15.v4_1_0m1778938175.8.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [16/May/2026 15:19:26] "GET /_dash-component-suites/dash/deps/polyfill@7.v4_1_0m1778938175.12.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [16/May/2026 15:19:26] "GET /_dash-component-suites/dash/dcc/dash_core_components.v4_1_0m1778938175.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [16/May/2026 15:19:26] "GET /_dash-component-suites/dash/dash-renderer/build/dash_renderer.v4_1_0m1778938175.min.js HTTP/1.1" 200 -
IN

**Важность признаков**

Согласно Explainer Dashboard, наибольшее влияние на предсказание модели оказывают следующие признаки:

ExitRates, BounceRates, PageValues, Month, SpecialDay, VisitorType.

Наиболее значимыми являются поведенческие характеристики пользователя: высокая доля выходов со страницы (ExitRates) и отказов (BounceRates) существенно снижает вероятность покупки.

При этом признак PageValues положительно влияет на прогноз: чем выше ценность посещаемых страниц, тем выше вероятность совершения покупки.

Также наблюдается влияние сезонных факторов (Month).

**Анализ качества модели**

значения метрик:

Accuracy ≈ 0.32
Precision ≈ 0.17
Recall ≈ 0.96
F1-score ≈ 0.30

Модель демонстрирует высокий recall, что означает способность находить большинство пользователей, совершивших покупку. Это важно в задачах, где критично не пропустить целевые события. Однако низкий precision указывает на большое количество ложноположительных прогнозов — модель часто ошибочно относит пользователей к покупателям. Причина такого поведения модели, вероятно, заключается в дисбалансе классов.

Анализ индивидуальных прогнозов
Пример 1: высокая вероятность покупки

Пользователь с:

низкими значениями ExitRates и BounceRates
высоким значением PageValues

Модель предсказывает высокую вероятность покупки, так как пользователь активно взаимодействует с сайтом и проявляет интерес к контенту.

Пример 2: низкая вероятность покупки

Пользователь с:

высокими ExitRates и BounceRates
низким PageValues

Вероятность покупки низкая, поскольку пользователь быстро покидает сайт и не вовлечён в просмотр.

Пример 3: пограничный случай

Пользователь со средними поведенческими характеристиками, но, например, в период повышенного спроса (определённый месяц).

Модель вероятность покупки выше за счёт влияния сезонного фактора.


Итоговые выводы:

1. Простые модели (Naive Bayes, kNN) показывают ограниченное качество на несбалансированных данных.
2. Подбор гиперпараметров даёт минимальный эффект без изменения структуры данных.
3. Добавление категориальных признаков значительно улучшает Recall, но ухудшает Precision.
4. Наиболее важными являются поведенческие признаки пользователя.
5. Модель склонна к переоценке положительного класса.



In [ ]:
html = db.to_html()

with open("dashboard.html", "w") as f:
    f.write(html)

In [ ]:

# эксперименты с другой моделью
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        class_weight='balanced',
        random_state=42
    ))
])

params_rf = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [5, 10, 20, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

from sklearn.model_selection import RandomizedSearchCV

rs_rf = RandomizedSearchCV(
    pipeline_rf,
    params_rf,
    n_iter=20,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    random_state=42
)

rs_rf.fit(X_train_full, y_train_full)

y_pred = rs_rf.best_estimator_.predict(X_test_full)

print_metrics(y_test_full, y_pred, "RandomForest (balanced)")


RandomForest (balanced):
Accuracy: 0.8936
Precision: 0.6353
Recall: 0.6998
F1-score: 0.6660
Confusion Matrix:
[[2406  186]
 [ 139  324]]
